In [ ]:
from datasets import load_dataset
import datasets
print(datasets.__version__)
print(datasets.__file__)
ds_hr = load_dataset("Polygl0t/Hellaswag-poly", "hr")
ds_en = load_dataset("Rowan/hellaswag")

In [ ]:
# Print the first 50 indices of the English validation set
for i, item in enumerate(ds_en["validation"]):
    if i >= 50:
        break
    print(item.get("ind", i))

In [ ]:
idx = 708  # or any index you want to inspect

# Find the corresponding index in English split
# Option 1: If both splits have the same length/order (just set idx_en = idx)
# Option 2: More robust: match 'ind' fields
en_item = next(x for x in ds_en["validation"] if x.get("ind", None) == idx) if "ind" in ds_en["validation"].features else ds_en["validation"][idx]
hr_item = next(x for x in ds_hr["validation"] if x["ind"] == idx)

print("ENGLISH EXAMPLE:")
print(en_item)
print("\n---\n")
print("CROATIAN EXAMPLE:")
print(hr_item)

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from pathlib import Path
import zipfile

# Download the actual dataset archive
zip_path = hf_hub_download(
    repo_id="classla/copa_hr",
    filename="data.zip",
    repo_type="dataset",
)

# Extract it
data_dir = Path("./data/copa_hr")
data_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(data_dir)

# Load without executing the legacy script
ds = load_dataset(
    "json",
    data_files={
        "train": str(data_dir / "train.jsonl"),
        "validation": str(data_dir / "val.jsonl"),
        "test": str(data_dir / "test.jsonl"),
    },
)

print(ds)
print(ds["validation"][0])

In [ ]:
from datasets import load_dataset

ds_en = load_dataset("aps/super_glue", "copa")

print(ds_en)
print(ds_en["train"][0])

In [ ]:
### eval COPA
import torch
from src.lightning.lit_gpt import LitGPT
import importlib

from datasets import load_dataset

ds_en = load_dataset("aps/super_glue", "copa")

print(ds_en)
print(ds_en["train"][0])

train_lm = importlib.import_module("src.scripts.gpt2-test-large.train_lm")
load_config = train_lm.load_config
from src.utils import paths

from src.models.lm import DecoderLM

language = "en"

exp_dir = paths.EXPERIMENTS_DIR / "best-gpt2-test-large/1024-dim-10M-token-20261012-late-recall"
#exp_dir = paths.EXPERIMENTS_DIR / "architecture-tests/all-changes-lr-5e-4-20261005-amicable-tangerine"
config_path = exp_dir / "config.resolved.yaml"
model_path = exp_dir / "last.ckpt"
#model_path = paths.EXPERIMENTS_DIR / "best-gpt2-test-large/fineweb-only-20261012-ternary-reef/lightning_logs/version_0/checkpoints/last.ckpt"
cfg = load_config(config_path)
transformer = DecoderLM(
    **cfg.model
)
lit = LitGPT.load_from_checkpoint(
    model_path,
    transformer=transformer,
    weights_only=False,
)
model = lit.transformer.eval()



In [ ]:
from src.tokenizers.BPE import BytePairEncoding
model.eval()

with torch.inference_mode():
    success_rate = 0 
    total = 0
    for id in range(400):
        total += 1
        ds = ds_en["train"]
        #print(ds[id]["premise"])
        #print(ds[id]["choice1"])
        #print(ds[id]["choice2"])

        # Use correct connector as in original logic
        connector_dict = {"cause": " because", "effect": " therefore"}
        connector = connector_dict[ds[id]["question"]]

        #print(connector)
        tokenizer = BytePairEncoding.from_file(paths.EXPERIMENTS_DIR / "gpt2-test-large/BPE_vocab_48k_hr_en")

        #print("correct " + str(ds[id]["label"] + 1))

        premise_text = ds[id]["premise"].strip()
        if premise_text.endswith("."):
            premise_text = premise_text[:-1]
        prompt = premise_text + connector

        # Lowercase first letter of each choice text before tokenizing, as in gpt2_original
        def lower_first(s):
            return s[:1].lower() + s[1:] if s else s

        premise = [tokenizer._special_vocab["<EN>"]] + tokenizer.tokenize(prompt)
        choice1 = tokenizer.tokenize(" " + lower_first(ds[id]["choice1"]))
        choice2 = tokenizer.tokenize(" " + lower_first(ds[id]["choice2"]))

        tokens_1 = premise + choice1
        tokens_2 = premise + choice2

        #print(tokenizer.decode(tokens_1))

        tokens_1 = torch.tensor(tokens_1, device="cuda").unsqueeze(0)
        #print(tokens_1.shape)
        tokens_2 = torch.tensor(tokens_2, device="cuda").unsqueeze(0)

        logits_1 = model.forward(tokens_1)
        logits_2 = model.forward(tokens_2)

        log_probs_1 = logits_1[:, len(premise)-1:-1, :].log_softmax(dim = -1)
        log_probs_2 = logits_2[:, len(premise)-1:-1, :].log_softmax(dim = -1)

        score_1 = log_probs_1.gather(-1, tokens_1[:, len(premise):].unsqueeze(-1)).squeeze(-1).sum() / len(choice1)
        score_2 = log_probs_2.gather(-1, tokens_2[:, len(premise):].unsqueeze(-1)).squeeze(-1).sum() / len(choice2)
        #print("choice_1 = ", score_1)
        #print("choice_2 = ", score_2)

        if ds[id]["label"] == 0:
            if score_1 > score_2:
                #print("success!")
                success_rate +=1
            #else:
                #print("fail!")
        else:
            if score_2 > score_1:
                #print("success")
                success_rate +=1
            #else:
                #print("fail")

    print(success_rate / total)

In [ ]:
0.5475 normalized by len   best-gpt2-test-large
0.5555 unnormalized

0.5775 / normalized 
0.5875 / unnormalized for finewebonly

0.58         normalized  768embedding dinm 20m tokens
0.6025            unnormalized

  0.6      normalized
0.625    unnormalized 1024embedding dim 10m tokens